**2_ Deeper Into Pytorch**

The goal of this assesment is to delve deeper in the fundamentals of pytorch.

The nice thing about pytorch is that implements different modules to allow us nice transformations over different types of data.

A **dataloader** is in charged of "asking" the dataset for new data and give this data to use it so that you can use it in your machine learning pipeline.

For this practice we will use Mnist dataset

In [2]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import gzip
import pickle
import imageio
import tempfile   
import os
from IPython import display
from io import BytesIO

In [3]:
#dataset directory

dataset_dir = "C:\\Users\\pablo\\Documents\\Desarrollo\\Practicas Redes Neuronales\\Datasets\\mnist.pkl.gz"

#Funcion para verificar que se abre
try:
    with gzip.open(dataset_dir, 'rb') as f:
        try:
            train_set, valid_set, test_set = pickle.load(f, encoding='latin1')
        except:
            train_set, valid_set, test_set = pickle.load(f)
except:
    raise ValueError("Download data")
## Read and concatenate all data into X and Y 
X_tr, Y_tr = train_set
X_va, Y_va = valid_set
X_te, Y_te = test_set

X = np.vstack((X_tr, X_va, X_te))
T = np.hstack((Y_tr, Y_va, Y_te))

In [4]:
# Now we will create the code for our dataset
class MNISTDataset(torch.utils.data.Dataset):
    def __init__(self, X,T, transforms = None):
        ## call parent method
        super().__init__()
        self.X = X
        self.T = T
        self.transforms = transforms
        
    def __len__(self):
        return len(self.T)
    
    def __getitem__(self, idx):
        X = self.X[idx]
        if self.transforms is not None:
            for t in self.transforms:
                X = t(X)
                
        return X, self.T[idx]
    

In [5]:
def reshape(X):       
    return np.reshape(X, (28,28))
    
mnist_dataset = MNISTDataset(X = X, T = T, transforms = [reshape])

**DATALOADER**

To grab images from our dataset, we use a dataloader, we will use a **batch_size** of 1, so each iteration we recieve one image.

In [6]:
%matplotlib tk


mnist_loader = torch.utils.data.DataLoader(
    dataset = mnist_dataset,
    batch_size = 1,
    shuffle = True,
    num_workers = 0,
    )

fig, ax = plt.subplots(1,1)

# Create temporary file for video creation
video_filename = os.path.join(tempfile.gettempdir(), "aux.mp4")

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

counter = 0

for x,y in mnist_loader:
    ax.clear()
    ax.imshow(x[0], cmap = 'gray')
    ax.set_title(f'Class {y.item()}')
    
    ## add frame for video creation
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)  

    if counter == 10:
        break
        
    counter += 1

writer.close() 
plt.close()

display.display(display.Video(data=video_filename, embed=True))
os.remove(video_filename)

C:\Users\pablo\AppData\Local\Temp\ipykernel_6500\308241461.py:31: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frame = imageio.imread(buf)


**Task 1: Implementing two custom transformations**

Our goal will be to create different versions of the mnist dataset that we will use to train convolutional models and to learn models using self-supervision, which is the fundamental machine learning technique

To make our transformation configurable we will create a class to change:
- RandomBlankSquare
- RandomRotation

In [7]:
from PIL import Image

def reshape(X):
    return np.reshape(X,(28,28))

class RandomBlankSquare:
    def __init__(self,min_h,max_h, min_v,max_v):
        self.min_h = min_h
        self.max_h = max_h
        self.min_v = min_v
        self.max_v = max_v
        
    def __call__(self, x):
        min_h_idx = np.random.randint(0, self.min_h, size=(1,), dtype = int).item()
        max_h_idx = np.random.randint(self.min_h, self.max_h, size=(1,), dtype = int).item()

        min_v_idx = np.random.randint(0,self.min_v, size=(1,), dtype = int).item()
        max_v_idx = np.random.randint(self.min_v, self.max_v, size=(1,), dtype = int).item()

        x[min_v_idx:max_v_idx, min_h_idx:max_h_idx] = 0

        return x

class RandomRotation:
    def __init__(self,max_degree):
        self.max_deg = max_degree
        
    def __call__(self, x):
        rot_deg = np.random.random(size=(1,)).item()*2*self.max_deg - self.max_deg 
        
        x = Image.fromarray(x)
        x = x.rotate(rot_deg)
        x = np.array(x)
        return x

In [8]:
# Now experiment in using these transformation


mnist_dataset = MNISTDataset(
                                X = X, 
                                T = T, 
                                transforms = [
                                            reshape, 
                                            RandomBlankSquare(min_h = 10, max_h = 20,min_v = 10, max_v = 20),
                                            RandomRotation(70),
                                        ]
                         )


mnist_loader = torch.utils.data.DataLoader(
                                                dataset = mnist_dataset,
                                                batch_size = 1, 
                                                shuffle=True,
                                                num_workers=0,
                                            )

# Create temporary file for video creation
video_filename = os.path.join(tempfile.gettempdir(), "aux.mp4")

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")
fig, ax = plt.subplots(1,1)

counter = 0
for x,y in mnist_loader:
    
    ax.imshow(x[0], cmap = 'gray')
    ax.set_title(f'Class {y.item()}')
    
    ## add frame for video creation
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    
    buf.seek(0)
    frame = imageio.imread(buf) 
    writer.append_data(frame)  
    
    if counter == 10:
        break

    counter += 1

writer.close() 
plt.close()


display.display(display.Video(data=video_filename, embed=True))
os.remove(video_filename)

C:\Users\pablo\AppData\Local\Temp\ipykernel_6500\3751360247.py:40: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frame = imageio.imread(buf)


**Task 2 Creating a pytorch dataset with our regressions problems**

We are going to create three datasets

In [9]:
class RtoR(torch.utils.data.Dataset):
    def __init__(self):
        ## call parent method
        super().__init__()
        self.X = np.array([0,1,2], dtype = np.float32).reshape(3,1)
        self.T = np.array([0.2,0.5,2.8], dtype = np.float32).reshape(3,1)
        
    def __len__(self):
        return len(self.T)
    
    def __getitem__(self, idx):
        return self.X[idx], self.T[idx]

class Rto01(torch.utils.data.Dataset):
    def __init__(self):
        ## call parent method
        super().__init__()
        self.X = np.array([-0.13459237,-3.3015387,0.74481176,2.62434536,0.38824359,0.47182825,-0.07296862], dtype = np.float32).reshape(7,1)
        self.T = np.array([0,0,0,1,1,1,1], dtype = np.float32).reshape(7,1)
    
    def __len__(self):
        return len(self.T)
    
    def __getitem__(self, idx):
        return self.X[idx], self.T[idx]

class R2to01NL(torch.utils.data.Dataset):
    def __init__(self):
        ## call parent method
        super().__init__()

        # input to our model. Represents time in seconds
        self.X = np.array([[0,1],
                           [1.5,2.0],
                           [2,1],
                           [5,3],
                           [3,4],
                           [4,5],
                           [5,1]], dtype = np.float32).reshape(7,2)

        # outputs associated to each input. 
        self.T =  np.array([0,0,0,0,1,1,1], dtype = np.float32).reshape(7,1)

    def __len__(self):
        return len(self.T)
    
    def __getitem__(self, idx):
        return self.X[idx], self.T[idx]

In [20]:
#loadindg the dataloaders  

rtor_loader = torch.utils.data.DataLoader(
    dataset = RtoR(),
    batch_size = len(RtoR()),
    shuffle = True,
    num_workers = 0,
)

rtor01_Loader = torch.utils.data.DataLoader(
    dataset=Rto01(),
    batch_size= len(Rto01()),
    shuffle=True,
    num_workers=0,
)

r2to01ml_Loader = torch.utils.data.DataLoader(
    dataset=R2to01NL(),
    batch_size= len(R2to01NL()),
    shuffle=True,
    num_workers=0,
)



**Torch nn module**

**torch.nn module** handles the boring parts, like tracking which parameters do you want to optimize, moving your model parameters to the GPU or switching the behaviour of different classes.

torch nn module has a base class, call module, that any model must inherit. In this case we are required to overwrite the forward method.

- **__init__**: this method initialize the computational 
- **forward**: this method is in charge of using the parameters to implement the set of operations that implements your desired function 
- **compute_loss**: additional method for loss computation


To train our model,  we need to perform our 5 operations:

1. Initialization
2. Forward
3. Backward
4. Update
5. Zero grad

To optimize our parameters it would be beneficial to add these parameters to a list that we can track.

Method that adds all our parameters to this list, so that we can trace them. T
- Method will be called def trace_parameter(self)
- Method that returns this parameter list def get_parameters(self) 


In [11]:
class LinearModel(nn.Module):
    def __init__(self, dim_in, dim_out, link_function, loss_function):
        super().__init__()
        
        ## create parameters
        self.W = torch.tensor(np.random.randn(dim_in, dim_out), requires_grad = True, dtype = torch.float32)
        self.b = torch.zeros(dim_out, requires_grad = True, dtype = torch.float32)
        
        ## trace parameters
        self.parameter_list = []
        self._trace_parameter(self.W)
        self._trace_parameter(self.b)
        
        ## link and loss funciton
        self.link = link_function
        self.loss = loss_function
        
    def forward(self,x, apply_link):
        y = torch.matmul(x, self.W) + self.b
    
        if apply_link:
            y = self.link(y)
        
        return y
    
    def compute_loss(self,t,y):
        return self.loss(y,t)
    
    def _trace_parameter(self,p):
        self.parameter_list.append(p)
    
    def get_parameters(self):
        return self.parameter_list
    
# We are going to create the first instance of our model, the link function is going to be the sigmoid function and the loss function is going to be the mean squared error.
model = LinearModel(dim_in=1, dim_out=1, link_function= torch.sigmoid, loss_function= torch.nn.MSELoss())

In [12]:
# Now we will train our model 
## ================= ##
## Training pipeline ##
## ================= ##
# Create temporary file for video creation
video_filename = os.path.join(tempfile.gettempdir(), "aux.mp4")

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

fig, (ax1, ax2) = plt.subplots(1,2)
N_points = 100
x_range = torch.linspace(-4, 4, N_points).reshape(N_points,1)
loss_acc_list = []

## create your data loader
ror_loader = torch.utils.data.DataLoader(
                                            dataset = RtoR(),
                                            batch_size = len(RtoR()), 
                                            shuffle=True,
                                            num_workers=0,
                                        )

## Model Initialization
def linear(x):
    '''Linear link function.'''
    return x
model = LinearModel(dim_in = 1, dim_out = 1, link_function = linear, loss_function = nn.MSELoss())

## optimization hyperparaemteres
epochs = 100
lr = 0.1
parameters = model.get_parameters()

## training loop
for e in range(epochs):
    loss_acc = 0.0
    for x, t in ror_loader:
        
        ## forward
        y = model(x, apply_link = True)
        L = model.compute_loss(t,y)
        loss_acc += L.item()
        
        ## backward
        L.backward()
        
        ## update
        for p in parameters:
            p.data = p.data - lr*p.grad.data
            
        ## zero grad
        for p in parameters:
            p.grad.data.zero_()
       
       
        print(f"On epoch {e} got loss {L}", end = "\r")
    
    ## plot regressed line and loss
    with torch.no_grad():
        ## add loss to loss list
        loss_acc_list.append(loss_acc)
        
        ax1.cla()
        ax2.cla()
        
        ax1.plot(x,t,'+')
        ax1.plot(x_range, model(x_range, apply_link = False))
        ax1.set_ylim([0,3])
        
        ax2.plot(np.arange(e+1), loss_acc_list)
        ax2.set_xlim([0,epochs])
    
        ## add frame for video creation
        buf = BytesIO()
        fig.savefig(buf, format="png", dpi=100)
        
        buf.seek(0)
        frame = imageio.imread(buf) 
        writer.append_data(frame) 

writer.close()
plt.close()

display.display(display.Video(data=video_filename, embed=True))
os.remove(video_filename)

C:\Users\pablo\AppData\Local\Temp\ipykernel_6500\2855985559.py:79: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frame = imageio.imread(buf)


#### **CREATION OF LINEAR MODULE AND IMPLEMENTED IN A LINEAR MODEL**

For more complex applicattion we will require additional methods that reset this parameter list or that are able to register parameters coming from a class implementing complex functionalities that your model wants to use. To achieve that we will:

1. Create a linear layer
2. Create a linear model using the previously created linaer module
3. We will inspect how PyTorch keeps track of the parameters in the module and model

In [13]:
# 1. Creation of the linear module

class Linear(nn.Module): 
    def __init__(self, dim_in, dim_out):    
        super().__init__()
        ## create parameters
        self.W = nn.Parameter(torch.tensor(np.random.randn(dim_in, dim_out), dtype = torch.float32))
        self.b = nn.Parameter(torch.zeros(dim_out, dtype = torch.float32))
    def forward(self, x):
        y = torch.matmul(x, self.W) + self.b
        return y    


# 2. Creation of the linear model

class LinearModel(nn.Module):
    def __init__(self, dim_in, dim_out, link_function, loss_function):
        super().__init__()
        self.layer = Linear(dim_in, dim_out)
        self.link = link_function
        self.loss = loss_function
        
    def forward(self,x, apply_link):
        y = self.layer(x)

        if apply_link:
            y = self.link(y)
        
        return y

    def compute_loss(self,t,y):
        return self.loss(y,t)

# 3. Checking the parameter management of the model

linear_layer = Linear(10,10)

for p in linear_layer.parameters():
    print(p.shape, " ", p.requires_grad)

for n,p in linear_layer.named_parameters():
    print(n , " ", p.shape, " ", p.requires_grad)

linear_layer = LinearModel(10,1, link_function = torch.sigmoid, loss_function = nn.MSELoss())

for p in linear_layer.parameters():
    print(p.shape, " ", p.requires_grad)

for n,p in linear_layer.named_parameters():
    print(n , " ", p.shape, " ", p.requires_grad)

torch.Size([10, 10])   True
torch.Size([10])   True
W   torch.Size([10, 10])   True
b   torch.Size([10])   True
torch.Size([10, 1])   True
torch.Size([1])   True
layer.W   torch.Size([10, 1])   True
layer.b   torch.Size([1])   True


In [ ]:
## ================= ##
## Training pipeline ##
## ================= ##
## to display learning functions
fig, (ax1, ax2) = plt.subplots(1,2)

# Create temporary file for video creation
video_filename = os.path.join(tempfile.gettempdir(), "aux.mp4")

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

x_range = torch.linspace(-4,4,100).reshape(100,1)
loss_acc_list = []

## create your data loader
rto01_loader = torch.utils.data.DataLoader(
                                            dataset = Rto01(),
                                            batch_size = len(Rto01()), 
                                            shuffle=True,
                                            num_workers=0,
                                         )

## Initialization
model = LinearModel(dim_in = 1, dim_out = 1, link_function = torch.sigmoid, loss_function = nn.BCEWithLogitsLoss() )

# move model to the desired device
device = 'cpu'
model.to(device) 

## optimization hyperparaemteres
epochs = 100
lr = 0.1
# grab parameters easily. Since it returns an iterator we need to keep it in a list because iterators can only be iterated once
parameters = [p for p in model.parameters()]

## training loop
for e in range(epochs):
    loss_acc = 0.0
    for x,t in rto01_loader:
        ## move to device
        x, t = x.to(device), t.to(device)
        
        ## forward
        y = model(x, apply_link = False)
        L = model.compute_loss(t,y)
        loss_acc += L.item()
        
        ## backward
        L.backward()
        
        ## update
        for p in parameters:
            p.data -= lr*p.grad
            
        ## zero grad
        model.zero_grad()
        
    print(f"On epoch {e} got loss {loss_acc}", end = "")
    
    ## plot regressed line and loss
    with torch.no_grad():
        ## add loss to loss list
        loss_acc_list.append(loss_acc)
        
        ax1.cla()
        ax2.cla()
        
        ax1.plot(x,t,'+')
        ax1.plot(x_range, model(x_range, apply_link = True))
        ax1.set_ylim([-0.5,1.5])
        
        ax2.plot(np.arange(e+1), loss_acc_list)
        ax2.set_xlim([0,epochs])
    
        ## add frame for video creation
        buf = BytesIO()
        fig.savefig(buf, format="png", dpi=100)
        
        buf.seek(0)
        frame = imageio.imread(buf) 
        writer.append_data(frame) 
  

writer.close()
plt.close()

display.display(display.Video(data=video_filename, embed=True))
os.remove(video_filename)

On epoch 0 got loss 0.5899321436882019On epoch 1 got loss 0.5837700963020325On epoch 2 got loss 0.5782503485679626On epoch 3 got loss 0.5732901692390442

C:\Users\pablo\AppData\Local\Temp\ipykernel_6500\3020079699.py:81: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frame = imageio.imread(buf)


On epoch 4 got loss 0.5688191056251526On epoch 5 got loss 0.5647769570350647On epoch 6 got loss 0.5611121654510498On epoch 7 got loss 0.5577806830406189On epoch 8 got loss 0.5547444820404053On epoch 9 got loss 0.5519705414772034On epoch 10 got loss 0.5494304299354553On epoch 11 got loss 0.5470993518829346On epoch 12 got loss 0.5449556708335876On epoch 13 got loss 0.5429805517196655On epoch 14 got loss 0.5411571264266968On epoch 15 got loss 0.5394710302352905On epoch 16 got loss 0.5379090905189514On epoch 17 got loss 0.5364599227905273On epoch 18 got loss 0.5351133346557617On epoch 19 got loss 0.5338602662086487On epoch 20 got loss 0.532692551612854On epoch 21 got loss 0.5316030979156494On epoch 22 got loss 0.5305851101875305On epoch 23 got loss 0.5296329855918884On epoch 24 got loss 0.5287414193153381On epoch 25 got loss 0.5279055833816528On epoch 26 got loss 0.5271211862564087On epoch 27 got loss 0.5263842344284058On epoch 28 got loss 0.5256913304328918On epoch 29 got loss 0.525039196

Definition of the las methods:

1. nn.Sequential()
2. nn.ParameterList
3. nn.ModuleList

In [27]:
def linear_activation(x):
    return x

class Layer(nn.Module):
    def __init__(self, dim_in, dim_out, act):
        super().__init__()
        ## create parameters
        self.linear = nn.Linear(dim_in, dim_out)
        self.act = act
        
    def forward(self, x):
        return self.act(self.linear(x))

class FCModuleList(nn.Module):
    def __init__(self, dim_in, dim_out, neurons_hidden : list, hidden_activations:list, link_function, loss_function):
        super().__init__()

        assert len(neurons_hidden) == len(hidden_activations), "List specifying hidden activations and number of hidden layers must coincide"

        module_list = nn.ModuleList([])

        # input layer hidden layers
        for num_neur, act in zip(neurons_hidden, hidden_activations):
            module_list.append(Layer(dim_in,num_neur,act))
            dim_in = num_neur

        # output layer
        o_layer = Layer(dim_in, dim_out, act = linear_activation)
        module_list.append(o_layer)

        self.layers = module_list
       
        ## Loss and link function
        self.link = link_function
        self.loss = loss_function
        
    def forward(self,x, apply_link):
        for l in self.layers:
            x = l(x)
        y = x
        if apply_link:
            y = self.link(y)
        
        return y
    def compute_loss(self,t,y):
        return self.loss(y,t)


class FCSequential(nn.Module):
    def __init__(self, dim_in, dim_out, neurons_hidden : list, hidden_activations:list, link_function, loss_function):
        super().__init__()

        assert len(neurons_hidden) == len(hidden_activations), "List specifying hidden activations and number of hidden layers must coincide"

        module_list = []

        # input layer hidden layers
        for num_neur, act in zip(neurons_hidden, hidden_activations):
            module_list.append(Layer(dim_in, num_neur, act))
            dim_in = num_neur

        # output layer
        o_layer = Layer(dim_in, dim_out, act = linear_activation)
        module_list.append(o_layer)

        self.layers_forward = nn.Sequential(*module_list)
       
        ## Loss and link function
        self.link = link_function
        self.loss = loss_function
        
    def forward(self,x, apply_link):
        y = self.layers_forward(x)
    
        if apply_link:
            y = self.link(y)

        return y

    def compute_loss(self,t,y):
        return self.loss(y,t)

### **TORCH OPTIMIZERS**

PyTorch optimizers are classes that implement different algorithms to train models. While basic gradient descent updates parameters in a simple way, more advanced optimizers (like SGD with momentum) improve convergence by using past gradient information. PyTorch provides these optimizers to avoid manual implementation, and they are initialized with the model’s parameters and hyperparameters such as the learning rate. After computing gradients with backward(), parameter updates and gradient resets are easily handled using opt.step() and opt.zero_grad().

In [29]:
## to display learning functions
fig, (ax1, ax2) = plt.subplots(1,2)

# Create temporary file for video creation
video_filename = os.path.join(tempfile.gettempdir(), "aux.mp4")
## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

fig, (ax1, ax2) = plt.subplots(1,2, figsize = (10,5))

loss_acc_list = []

N_points_domain = 100
thr_prob = 0.5 # use to plot our classification guess
x1, x2 = np.meshgrid(np.linspace(-2,7,N_points_domain, dtype = np.float32),np.linspace(-2,7,N_points_domain, dtype = np.float32))

# reshape for neural network
x_range = torch.from_numpy(
    np.hstack((np.reshape(x1, (N_points_domain**2,1)),np.reshape(x2, (N_points_domain**2,1))))
).float()

# allocate memory to plot decision thresholds
y_range_plot = np.zeros((N_points_domain,N_points_domain), np.float32)

# class colors
color_c0 = 'C0'
color_c1 = 'C1'

## ======= DATA ==========
## create your data loader
r2to01nl_loader = torch.utils.data.DataLoader(
                                            dataset = R2to01NL(),
                                            batch_size = len(R2to01NL()), 
                                            shuffle=False,
                                            num_workers=0,
                                         )

## Initialization
model = FCModuleList(
    dim_in = 2, 
    dim_out = 1, 
    neurons_hidden = [10,10], 
    hidden_activations = [torch.relu, torch.relu], 
    link_function = torch.sigmoid, 
    loss_function = nn.BCEWithLogitsLoss()
)
# move model to the desired device
device = 'cpu'
model.to(device) 
model.float()

## optimization hyperparaemteres
epochs = 100
optimizer = torch.optim.SGD(params=model.parameters(),lr = 0.1, momentum = 0.9)

for e in range(epochs):
    loss_acc = 0.0
    for x, t in r2to01nl_loader:
        
        ## move to your computing platform
        x, t = x.to(device), t.to(device)
        
        ## 2.forward
        y = model(x, apply_link = False)
        L = model.compute_loss(t,y)
        
        # 3. Backward
        L.backward()
        loss_acc += L.item()
        
        # 4 Parameter update
        optimizer.step()
        
        # 5 Zero grad
        optimizer.zero_grad()

        print(f"On epoch {e} got loss {loss_acc}", end = "")
        
        with torch.no_grad():
            if e == 0:
                idx_class0 = (t == 0).squeeze()
                idx_class1 = (t == 1).squeeze()
                
            loss_acc_list.append(loss_acc)
            
            ## forward to plot decision thresholds
            y_range = model(x_range, apply_link = True)

            # reshape back to plotting
            y_range = np.reshape(y_range, (N_points_domain,N_points_domain))
            
            # start plotting
            ax1.cla()
            ax2.cla()
    
            # plot dataset    
            ax1.plot(x[idx_class0][:,0],x[idx_class0][:,1],'o', color = color_c0, markersize = 8, label = r'observations class 0 cat')
            ax1.plot(x[idx_class1][:,0],x[idx_class1][:,1],'*', color = color_c1,markersize = 8, label = r'data observations class 1dog')
            ax1.set_xlabel('peso ($x_1$)')
            ax1.set_ylabel('altura ($x_2$)')
    
            ## plot prediction probability for class 1 and 0
            idx_range1 = y_range > thr_prob
            idx_range0 = ~idx_range1
    
            y_range_plot[idx_range1] = y_range[idx_range1]
            y_range_plot[idx_range0] = np.nan
    
            ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Oranges"), alpha = 0.5)
            contourf1 = ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Oranges"), alpha = 0.5)
    
            y_range_plot[idx_range0] = y_range[idx_range0]
            y_range_plot[idx_range1] = np.nan
    
            ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Blues"), alpha = 0.5)
            contourf2 = ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Blues"), alpha = 0.5)
    
            # decision threshold
            contour1 = ax1.contour(x1, x2, y_range, levels = [thr_prob], colors = ["k"], linewidth = 4, label = f'decision threshold p = {thr_prob}')
            ax1.clabel(contour1, inline=True, fontsize=8, fmt="%.2f")
    
            ## set legend
            ax1.legend()
    
            ## set contour bar level
            if e == 0:
                cbar2 = fig.colorbar(contourf2, ax=ax1, orientation='vertical')
                cbar2.set_label('Probability of being a cat')
    
                #cbar1 = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # Ajusta la posición [izq, abajo, ancho, alto]
                cbar1 = fig.colorbar(contourf1, ax=ax1, orientation='vertical')
                cbar1.set_label('Probability of being a dog')
    
            ax2.plot(np.arange(0,e+1),loss_acc_list)
            ax2.set_xlim([0,epochs])
            ax2.set_xlabel('epochs')
            ax2.set_ylabel('loss')

            ## add frame for video creation
            buf = BytesIO()
            fig.savefig(buf, format="png", dpi=100)
            
            buf.seek(0)
            frame = imageio.imread(buf) 
            writer.append_data(frame) 

writer.close()
plt.close()

display.display(display.Video(data=video_filename, embed=True))
os.remove(video_filename)

C:\Users\pablo\AppData\Local\Temp\ipykernel_6500\3194659397.py:109: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Oranges"), alpha = 0.5)
C:\Users\pablo\AppData\Local\Temp\ipykernel_6500\3194659397.py:110: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  contourf1 = ax1.contourf(x1, x2, y_range_plot, levels = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1], cmap = plt.cm.get_cmap("Oranges"), alpha = 0.5)
C:\Users\pablo\AppData\Local\Temp\ipykernel_6500\3194659397.py:115: MatplotlibDeprecationWarning: The get_cmap function was deprecated

On epoch 0 got loss 0.6934904456138611On epoch 1 got loss 0.6903734803199768On epoch 2 got loss 0.6845436692237854On epoch 3 got loss 0.676396906375885On epoch 4 got loss 0.6662744283676147On epoch 5 got loss 0.654485285282135On epoch 6 got loss 0.6419744491577148On epoch 7 got loss 0.6281312108039856On epoch 8 got loss 0.6131185293197632On epoch 9 got loss 0.5971540212631226On epoch 10 got loss 0.5797609686851501On epoch 11 got loss 0.5612931847572327On epoch 12 got loss 0.5436003804206848On epoch 13 got loss 0.5267015099525452On epoch 14 got loss 0.5119887590408325On epoch 15 got loss 0.49594587087631226On epoch 16 got loss 0.478103905916214On epoch 17 got loss 0.4580928385257721On epoch 18 got loss 0.4414096176624298On epoch 19 got loss 0.42437928915023804On epoch 20 got loss 0.4139311909675598On epoch 21 got loss 0.4004310071468353On epoch 22 got loss 0.3820924162864685On epoch 23 got loss 0.36119356751441956On epoch 24 got loss 0.34944650530815125On epoch 25 got loss 0.33139827847